# TranslateGemma 12B + adaptador de latín (rama `modelo-12b`)

En la primera corrida del 12B, la mejor configuración fue **6 candidatos + umbral 0.85** (similitud media 0.903 contra 0.866
del 4B con el v2, en las mismas 5 frases), pero **sin adaptador**: los de `adaptador_latin/` y `adaptador_latin_v2/` son
del 4B y no le sirven al 12B. Aquí se entrena uno para el 12B y se repite esa configuración con él.

- **QLoRA:** el 12B se queda comprimido a 4 bits (~8.5 GB) y se entrenan solo las matrices LoRA (r = 16, atención q/k/v/o),
  igual que el v1. Cabe en la T4 (~11-13 GB en total).
- **Datos:** 600 pares de OPUS, mitad frases cortas (las del v1) y mitad largas de 8-22 palabras (las del v2), en las dos
  direcciones = 150 pasos.
- **Se guarda en tu Drive cada 10 pasos** (`MyDrive/traductor_12b/adaptador_latin_12b/`). Si Colab se desconecta, vuelve
  a dar *Ejecutar todas*: sigue desde el último guardado. Si ya terminó, se salta el entrenamiento.

**Antes de empezar:** `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)` y luego `Ejecutar todas`.
Te va a pedir permiso para conectar Drive: **acepta y marca todas las casillas**.

| Paso | Tiempo aprox. en T4 |
|---|---|
| Instalar y descargar el 12B (24 GB) | 10-15 min |
| Entrenar el adaptador (150 pasos) | ~1.5-2 h (estimado; imprime cuánto falta cada 20 pasos) |
| Comparar 10 frases con 6 candidatos + umbral 0.85 | ~40 min |

Al final se **descarga `resultados_12b_adaptador.zip`**.

## 1. Verificar GPU

In [ ]:
import torch
print("GPU disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ No hay GPU. Ve a Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU.")

## 2. Obtener el código (rama `modelo-12b`) e instalar librerías

In [ ]:
REPO = "AndreaHortaHernandez/Traductormultilingue"
RAMA = "modelo-12b"

import os
if not os.path.isdir("/content/repo"):
    !git clone -b {RAMA} https://github.com/{REPO}.git /content/repo
else:
    !cd /content/repo && git fetch origin {RAMA} && git checkout {RAMA} && git pull origin {RAMA}
%cd /content/repo
!git log --oneline -1
!pip install -q -U transformers sentence-transformers sacrebleu peft bitsandbytes gradio
!pip uninstall -q -y torchao

## 3. Guardar el adaptador en Drive

`adaptador_latin_12b/` del repo apunta a una carpeta de tu Drive, así lo entrenado no se pierde si Colab se desconecta.

In [ ]:
from google.colab import drive
try:
    drive.mount("/content/drive")
    DRIVE_ADAPTADOR = "/content/drive/MyDrive/traductor_12b/adaptador_latin_12b"
    EN_DRIVE = True
except Exception as error:
    # Pasa si la ventana de permisos se cierra, se rechaza o no se marcan todas las casillas.
    # Se sigue sin Drive: funciona igual, pero si Colab se desconecta se pierde lo entrenado.
    print(f"⚠️ No se pudo conectar Drive ({error}). Vuelve a correr esta celda y acepta TODOS los permisos;")
    print("   si sigue fallando, el adaptador se guarda solo en Colab (no se puede continuar si se desconecta).")
    DRIVE_ADAPTADOR = "/content/adaptador_latin_12b_local"
    EN_DRIVE = False
os.makedirs(DRIVE_ADAPTADOR, exist_ok=True)
if not os.path.islink("/content/repo/adaptador_latin_12b"):
    os.symlink(DRIVE_ADAPTADOR, "/content/repo/adaptador_latin_12b")

import json
ruta_estado = os.path.join(DRIVE_ADAPTADOR, "estado.json")
if os.path.exists(ruta_estado):
    e = json.load(open(ruta_estado))
    print(f"Ya hay un adaptador en el Drive: paso {e['paso']} de {e['total']}" + (" (terminado)" if e["terminado"] else " (se continúa)"))
else:
    print("No hay adaptador todavía: se entrena desde cero.")

## 4. Entrenar el adaptador de latín del 12B

No cambies `PARES` a mitad de un entrenamiento: el número de pasos cambia y no se puede continuar.

In [ ]:
ENTRENAR = True  #@param {type:"boolean"}
PARES = 600  #@param {type:"integer"}

import gc, time, config
config.PARES_12B = PARES
if ENTRENAR:
    import afinar_latin
    inicio = time.time()
    afinar_latin.afinar_12b()
    print(f"({(time.time() - inicio) / 60:.0f} min)")
gc.collect(); torch.cuda.empty_cache()   # libera la GPU antes de cargar el traductor

if os.path.exists(ruta_estado):
    e = json.load(open(ruta_estado))
    print("Pérdida cada 20 pasos:", e["perdidas"])

## 5. Cargar el 12B con el adaptador y LaBSE

In [ ]:
from modelos import Traductor, Verificador
inicio = time.time()
traductor = Traductor()
verificador = Verificador()
print(f"\n{config.NOMBRE_MODELO} listo en {(time.time() - inicio) / 60:.1f} min | con adaptador: {traductor.con_adaptador}")
print(f"Memoria de GPU en uso: {torch.cuda.memory_allocated() / 1e9:.1f} GB de "
      f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 6. Probar una frase (6 candidatos + umbral 0.85)

In [ ]:
frase = "Aunque estaba muy cansado, siguió trabajando hasta la medianoche."  #@param {type:"string"}
cadena = "la ko hi la ko hi"  #@param {type:"string"}

from cadena import traducir_en_cadena, imprimir_pasos
inicio = time.time()
imprimir_pasos(frase, traducir_en_cadena(frase, cadena.split(), traductor, verificador, num_candidatos=6, umbral=0.85))
print(f"\n({time.time() - inicio:.0f} s)")

## 7. Comparar con las mismas 10 frases complejas

Mismas frases y cadena que la corrida anterior del 12B, para ponerlas lado a lado:
**12B + 6 cand. + umbral 0.85** sin adaptador (corrida anterior) contra **con adaptador** (esta).
Marca `TAMBIEN_NORMAL` para agregar el 12B con adaptador y la configuración normal (~+25 min).

In [ ]:
COMPARAR = True  #@param {type:"boolean"}
TAMBIEN_NORMAL = False  #@param {type:"boolean"}
limite = 10  #@param {type:"integer"}

if COMPARAR:
    if not traductor.con_adaptador:
        print("⚠️ No se cargó el adaptador: termina primero el entrenamiento (sección 4).")
    else:
        import comparar
        variantes = comparar.VARIANTES_12B_ADAPTADOR + (["12B + adaptador"] if TAMBIEN_NORMAL else [])
        comparacion = comparar.comparar(traductor, verificador, ruta_frases="frases_complejas.txt",
                                        limite=limite, variantes=variantes)
        !cp adaptador_latin_12b/estado.json salidas/estado_adaptador_12b.json
        if EN_DRIVE:
            !mkdir -p /content/drive/MyDrive/traductor_12b && cp -r salidas /content/drive/MyDrive/traductor_12b/
            !cd /content/repo && zip -qr resultados_12b_adaptador.zip salidas
        else:   # sin Drive, el adaptador va dentro del zip para no perderlo
            !cd /content/repo && zip -qr resultados_12b_adaptador.zip salidas adaptador_latin_12b/
        from google.colab import files
        files.download("/content/repo/resultados_12b_adaptador.zip")